# AD GWAS gene loci across all studies (data processing)

Fig 6-style locus tracks for selected genes in **every** AD GWAS study: the LD block holding each gene is the fine-mapping block containing its TSS (`gene_cis_TADB_mapper.tsv`), then `load_locus()` (`R/utils.R`) loads the GWAS −log10 p, SuSiE-RSS and SuSiE-rRSS PIPs / CSs and the AD xQTL evidence for each study with a fit in that block, plus the 2026 AD GWAS colocalization (TransMap) rows.

SuSiE-rRSS is the full-pipeline result: the SER model replaces the fit when the SER fallback was used (`serFallbackUsed` in `export/SER_flag.tsv`, else the fit's own reliability flag), as in the APOE figure.

**Outputs:**
- `figure_data/AD_GWAS_Gene_Loci_All_Studies.rds` — named list by gene: `gene`, `tss`, `bid`, `loci` (named by study; each = `load_locus()` output + `tm` + `ser_fallback`)
- `figure_data/AD_GWAS_Gene_Loci_CS_Counts.csv` — gene × study: credible sets under each method and the SER fallback flag

In [1]:
source("../../R/utils.R")
library(pecotmr)
devtools::load_all("/hpc/users/sunh17/haochen/rss_qc/pecotmr_latest/pecotmr", compile = FALSE)   # getSumStatsDf()
library(readxl)
filter <- dplyr::filter; select <- dplyr::select

genes <- c("CLU", "SLC24A4", "ADAM10", "APH1B", "ABCA7", "ALPK2")
gene_mapper_file <- "/hpc/users/sunh17/haochen/rss_qc/gene_cis_TADB_mapper.tsv"   # gene TSS (+ cis window, TAD block)

root      <- "/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10"
susie_sub <- "AD_GWAS_rss_baseline_r4_L10"
ebmix_sub <- "AD_GWAS_rss_ebmix_r4_L10"

ℹ Loading pecotmr
Creating a generic function for ‘endoapply’ from package ‘S4Vectors’ in package ‘pecotmr’

Warning message:
“Failed to load at least one DLL.
Caused by error in `getDLLRegisteredRoutines.DLLInfo()`:
! must specify DLL via a “DLLInfo” object. See getLoadedDLLs()”


In [2]:
### AD GWAS explorer data, 2026 AD GWAS colocalization, SER fallback flags
AD_loci <- suppressMessages(suppressWarnings(
  read_excel("/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/unified_AD_loci_xQTL_summary.xlsx", skip = 2)))

transmap_long <- read_tsv(file.path(root, ebmix_sub, "export/transmap_AD_target_results_pv1e4.tsv"),
                          show_col_types = FALSE) %>% tidy_transmap()

ser_flag <- read_tsv(file.path(root, ebmix_sub, "export/SER_flag.tsv"), show_col_types = FALSE) %>%
  transmute(study, block_id, ser_fallback = serFallbackUsed %in% TRUE)

## Gene → LD block

In [3]:
## gene TSS from the cis / TAD block mapper
gene_pos <- read_tsv(gene_mapper_file, show_col_types = FALSE) %>%
  rename(chr = `#chr`) %>%
  filter(gene_name %in% genes) %>%
  transmute(gene = gene_name, gene_id, chr = str_remove(chr, "^chr"), tss = TSS, TADB_id) %>%
  distinct(gene, .keep_all = TRUE)
gene_pos
stopifnot(setequal(gene_pos$gene, genes))

## every study x block with a SuSiE-rRSS fit, and the blocks' coordinates
fits <- tibble(unit = list.files(file.path(root, ebmix_sub, "output/fine_mapping"), pattern = "\\.gwas_finemap\\.rds$")) %>%
  mutate(unit     = str_remove(unit, "\\.gwas_finemap\\.rds$"),
         study    = str_split_fixed(unit, "\\.", 2)[, 1],
         block_id = str_split_fixed(unit, "\\.", 2)[, 2])
blocks <- distinct(fits, block_id) %>%
  mutate(m = str_match(block_id, "^chr([0-9XY]+)_([0-9]+)_([0-9]+)$"),
         chr = m[, 2], bstart = as.numeric(m[, 3]), bend = as.numeric(m[, 4])) %>%
  select(-m) %>% filter(!is.na(chr))

## fine-mapping blocks containing the gene TSS
gene_block_all <- gene_pos %>%
  inner_join(blocks, by = "chr", relationship = "many-to-many") %>%
  filter(bstart <= tss, tss <= bend) %>%
  left_join(count(fits, block_id, name = "n_studies"), by = "block_id")
gene_block_all %>% select(gene, chr, tss, TADB_id, block_id, n_studies)

## one block per gene: where an original and a combined block both hold it, the one fitted in most studies
gene_block <- gene_block_all %>%
  group_by(gene) %>% slice_max(n_studies, n = 1, with_ties = FALSE) %>% ungroup() %>%
  mutate(gene = factor(gene, levels = genes)) %>% arrange(gene) %>% mutate(gene = as.character(gene))
gene_block %>% select(gene, block_id, n_studies)
stopifnot(nrow(gene_block) == length(genes))

gene,gene_id,chr,tss,TADB_id
<chr>,<chr>,<chr>,<dbl>,<chr>
SLC24A4,ENSG00000140090,14,92322580,chr14_89256526_92780707
ADAM10,ENSG00000137845,15,58749790,chr15_54171378_59497866
APH1B,ENSG00000138613,15,63276017,chr15_58574103_63343138
ALPK2,ENSG00000198796,18,58629090,chr18_55651342_58738603
ABCA7,ENSG00000064687,19,1040106,chr19_0_6791225
CLU,ENSG00000120885,8,27614699,chr8_24141394_29490278


gene,chr,tss,TADB_id,block_id,n_studies
<chr>,<chr>,<dbl>,<chr>,<chr>,<int>
SLC24A4,14,92322580,chr14_89256526_92780707,chr14_92293029_94303139,12
ADAM10,15,58749790,chr15_54171378_59497866,chr15_58149891_59314387,12
APH1B,15,63276017,chr15_58574103_63343138,chr15_61125463_66680537,12
ALPK2,18,58629090,chr18_55651342_58738603,chr18_58180232_59563580,12
ABCA7,19,1040106,chr19_0_6791225,chr19_90949_1507536,12
CLU,8,27614699,chr8_24141394_29490278,chr8_26225312_29469590,12


gene,block_id,n_studies
<chr>,<chr>,<int>
CLU,chr8_26225312_29469590,12
SLC24A4,chr14_92293029_94303139,12
ADAM10,chr15_58149891_59314387,12
APH1B,chr15_61125463_66680537,12
ABCA7,chr19_90949_1507536,12
ALPK2,chr18_58180232_59563580,12


## Loci: every study with a fit in each block

In [4]:
get_fit <- function(sub, sid, bid)
  attributes(readRDS(file.path(root, sub, "output/fine_mapping", paste0(sid, ".", bid, ".gwas_finemap.rds"))))$elementMetadata$susieFit[[1]]

## SuSiE-rRSS pipeline result: the SER model replaces the fit when the fallback was used
rrss_pipeline <- function(loc) {
  fit  <- get_fit(ebmix_sub, loc$sid, loc$bid)
  fl   <- filter(ser_flag, study == loc$sid, block_id == loc$bid)
  flag <- if (nrow(fl)) any(fl$ser_fallback) else isTRUE(fit$R_finite_diagnostics$R_reliability_flag)
  ser  <- fit$R_finite_diagnostics$ser_model
  if (flag && !is.null(ser)) {
    if (is.null(dim(ser$alpha))) ser$alpha <- matrix(ser$alpha, nrow = 1)
    colnames(ser$alpha) <- colnames(fit$alpha)
    loc$pip_df <- bind_rows(filter(loc$pip_df, method != "SuSiE-rRSS"), locus_tidy_fit(ser, "SuSiE-rRSS"))
  }
  loc$ser_fallback <- flag
  loc
}

gene_loci <- pmap(gene_block, function(gene, tss, block_id, ...) {
  studies <- fits %>% filter(block_id == !!block_id) %>% pull(study) %>% sort()
  loci <- map(setNames(studies, studies), function(sid)
    tryCatch(load_locus(block_id, sid, AD_loci, root = root, susie_sub = susie_sub, ebmix_sub = ebmix_sub) %>%
               rrss_pipeline() %>% { .$tm <- locus_transmap(transmap_long, .$reg); . },
             error = function(e) { message("FAILED: ", sid, ".", block_id, " : ", conditionMessage(e)); NULL }))
  list(gene = gene, tss = tss, bid = block_id, loci = compact(loci))
}) %>% set_names(gene_block$gene)

map_int(gene_loci, ~ length(.x$loci))       # studies loaded per gene

No z in sumstats for AD_Bellenguez_EADI_2022.chr14_92293029_94303139; GWAS p track left empty

FAILED: AD_Bellenguez_EADI_2022.chr14_92293029_94303139 : ℹ In argument: `chr == as.character(reg$chr)`.
Caused by error in `h()`:
! error in evaluating the argument 'x' in selecting a method for function 'colnames': subscript is out of bounds

No z in sumstats for AD_Bellenguez_GRACE_2022.chr14_92293029_94303139; GWAS p track left empty

FAILED: AD_Bellenguez_GRACE_2022.chr14_92293029_94303139 : ℹ In argument: `chr == as.character(reg$chr)`.
Caused by error in `h()`:
! error in evaluating the argument 'x' in selecting a method for function 'colnames': subscript is out of bounds

No z in sumstats for AD_Bellenguez_EADI_2022.chr15_58149891_59314387; GWAS p track left empty

FAILED: AD_Bellenguez_EADI_2022.chr15_58149891_59314387 : ℹ In argument: `chr == as.character(reg$chr)`.
Caused by error in `h()`:
! error in evaluating the argument 'x' in selecting a method for function 'colnames': subscr

CLU SLC24A4  ADAM10   APH1B   ABCA7   ALPK2 
     12      10      11      10      11       6

### credible sets per gene x study and method, and the SER fallback flag


In [5]:
cs_counts <- imap_dfr(gene_loci, function(g, gene) imap_dfr(g$loci, function(loc, sid)
  loc$pip_df %>% filter(cs != "0") %>% distinct(method, cs) %>% count(method) %>%
    complete(method = c("SuSiE-RSS", "SuSiE-rRSS"), fill = list(n = 0L)) %>%
    pivot_wider(names_from = method, values_from = n) %>%
    mutate(gene = gene, block_id = g$bid, study = sid, study_label = unname(cohort_labels[sid]),
           ser_fallback = loc$ser_fallback, .before = 1)))
as.data.frame(cs_counts)

save_data(gene_loci, "AD_GWAS_Gene_Loci_All_Studies")
write_csv(cs_counts, file.path(data_dir, "AD_GWAS_Gene_Loci_CS_Counts.csv"))

gene,block_id,study,study_label,ser_fallback,SuSiE-rRSS,SuSiE-RSS
<chr>,<chr>,<chr>,<chr>,<lgl>,<int>,<int>
CLU,chr8_26225312_29469590,AD_Bellenguez_2022,Bell.,FALSE,2,2
CLU,chr8_26225312_29469590,AD_Bellenguez_EADB_2022,Bell. EADB,FALSE,1,1
CLU,chr8_26225312_29469590,AD_Bellenguez_EADI_2022,Bell. EADI,FALSE,0,0
CLU,chr8_26225312_29469590,AD_Bellenguez_GRACE_2022,Bell. GRACE,FALSE,0,0
CLU,chr8_26225312_29469590,AD_Bellenguez_main_2026,EADB. 2026,FALSE,2,3
CLU,chr8_26225312_29469590,AD_Bellenguez_nobiobank_2026,EADB. 2026 -biobank,FALSE,2,3
CLU,chr8_26225312_29469590,AD_Bellenguez_noproxy_2026,EADB. 2026 -proxy,FALSE,2,2
CLU,chr8_26225312_29469590,AD_Jansen_2021,Jansen,FALSE,2,2
CLU,chr8_26225312_29469590,AD_Kunkle_Stage1_2019,Kunkle,FALSE,1,1
